# Projeto LusoEnergia — Parte III: Previsão e Seleção de Variáveis sem Fuga Temporal

Este notebook implementa integralmente os requisitos da **Parte III**:
1. **Problema Preditivo e Engenharia de Variáveis Causais (`3.1`)**:
   - Formulação da previsão do **consumo energético total do dia seguinte** (`Energia_Total_kWh`, em kWh/dia).
   - Construção de **22 variáveis candidatas** ($\ge 12$ exigidas): *lags* autoregressivos (`1, 2, 3, 7` dias), *lags* dos sub-contadores e consumo variável acima da carga basal, estatísticas móveis (`3, 7, 14` dias) e variáveis determinísticas de calendário (dia da semana, fim de semana, mês, codificação cíclica e feriados nacionais via biblioteca `holidays`).
   - Garantia rigorosa de **ausência de fuga temporal (*no data leakage*)**: todas as medições utilizam deslocamento temporal estrito ($\le t-1$).
2. **Seleção Comparada dentro de `Pipeline` e Validação Temporal (`3.2`)**:
   - Comparação de três paradigmas de seleção de variáveis — **Filter** (`SelectKBest` com Informação Mútua), **Wrapper** (`RFE` com Regressão Linear) e **Embedded** (`SelectFromModel` com `LassoCV` temporal) — todos encapsulados em `sklearn.pipeline.Pipeline` dentro de `TimeSeriesSplit(n_splits=5)`.
   - Justificação formal do porquê de `KFold` aleatório introduzir *data leakage* em séries temporais.
   - Avaliação comparativa da **Regressão Linear** com **todas as variáveis** vs. **subconjuntos selecionados** (MAE, RMSE e $R^2$) e diagnóstico de **estabilidade da seleção entre folds** (frequência e índice de Jaccard).
   - Recomendação prática à **LusoEnergia** sobre quais variáveis medir e manter em produção.

In [ ]:
import os
import warnings
from functools import partial
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import holidays

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_regression, RFE, SelectFromModel
from sklearn.linear_model import LinearRegression, LassoCV
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams['figure.dpi'] = 110

## 3.1 Problema Preditivo e Construção de Variáveis Candidatas (Sem Fuga Temporal)

### Garantia da Linha Temporal (Regra Causal)
Para prever o consumo total de um dado dia $t$ (`Energia_Total_kWh[t]`), a previsão tem de ser emitida **antes do início do dia $t$** (isto é, com base apenas na informação acumulada até às 23:59 do dia $t-1$):
- **Variáveis de consumo e sensores:** Todas calculadas com `.shift(1)` ou superior (`lag_1`, `lag_2`, `lag_3`, `lag_7` e janelas móveis aplicadas sobre a série já deslocada de 1 dia). Nenhuma medição do próprio dia $t$ entra nas variáveis explicativas.
- **Variáveis de calendário:** O dia da semana, o mês e o estatuto de feriado do dia $t$ são conhecidos deterministicamente com antecedência, não constituindo fuga de informação.
- **Tratamento de dias sem leitura:** Os dias completos sem medições identificados na Parte I não são imputados artificialmente; as linhas afetadas por `NaN` são removidas após a construção dos *lags* para que o treino e a avaliação incidam exclusivamente sobre consumo real.

In [ ]:
# 1. Janela temporal do grupo (18 meses)
SOMA_DOS_NUMEROS_DE_ALUNO = 101833
rng = np.random.default_rng(SOMA_DOS_NUMEROS_DE_ALUNO)
inicio = pd.Timestamp(rng.choice(pd.date_range('2007-01-01', '2009-05-01', freq='MS')))
fim = inicio + pd.DateOffset(months=18)

# 2. Carregamento ao minuto
possiveis_caminhos = [
    '../data/raw/individual+household+electric+power+consumption.zip',
    'data/raw/individual+household+electric+power+consumption.zip',
    '../data/raw/household_power_consumption.zip',
    'data/raw/household_power_consumption.zip'
]
caminho_zip = next((p for p in possiveis_caminhos if os.path.exists(p)), possiveis_caminhos[0])

dtypes = {
    'Date': 'str', 'Time': 'str',
    'Global_active_power': 'float32', 'Global_reactive_power': 'float32',
    'Voltage': 'float32', 'Global_intensity': 'float32',
    'Sub_metering_1': 'float32', 'Sub_metering_2': 'float32', 'Sub_metering_3': 'float32'
}

df = pd.read_csv(caminho_zip, sep=';', na_values=['?'], dtype=dtypes, low_memory=False, compression='zip')
df['datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], format='%d/%m/%Y %H:%M:%S')
df.drop(columns=['Date', 'Time'], inplace=True)
df.set_index('datetime', inplace=True)
df = df.loc[inicio:fim].copy()

# Interpolar apenas lacunas curtas (<= 60 min), mantendo dias completos em falta como NaN
df_min = df.interpolate(method='time', limit=60)

# Carga basal constante estimada na Parte I (P10)
CARGA_BASAL_KW = float(df_min['Global_active_power'].quantile(0.10))
ENERGIA_BASAL_KWH_DIA = CARGA_BASAL_KW * 24.0

# Energia por minuto em Wh
df_min['Energia_Total_Wh'] = (df_min['Global_active_power'] * 1000.0) / 60.0
df_min['Sub_outros_Wh'] = (
    df_min['Energia_Total_Wh'] - (df_min['Sub_metering_1'] + df_min['Sub_metering_2'] + df_min['Sub_metering_3'])
).clip(lower=0)

# 3. Agregação diária (exigindo pelo menos 1200 minutos válidos = 20h para considerar o dia completo)
df_diario = pd.DataFrame(index=df_min.resample('D').mean().index)
df_diario['Energia_Total_kWh'] = df_min['Energia_Total_Wh'].resample('D').sum(min_count=1200) / 1000.0
df_diario['Sub1_Cozinha_kWh'] = df_min['Sub_metering_1'].resample('D').sum(min_count=1200) / 1000.0
df_diario['Sub2_Lavandaria_kWh'] = df_min['Sub_metering_2'].resample('D').sum(min_count=1200) / 1000.0
df_diario['Sub3_Climatizacao_kWh'] = df_min['Sub_metering_3'].resample('D').sum(min_count=1200) / 1000.0
df_diario['Sub_Outros_kWh'] = df_min['Sub_outros_Wh'].resample('D').sum(min_count=1200) / 1000.0
df_diario['Energia_Variavel_kWh'] = (df_diario['Energia_Total_kWh'] - ENERGIA_BASAL_KWH_DIA).clip(lower=0)
df_diario['Voltage_Media'] = df_min['Voltage'].resample('D').mean()
df_diario['Reactive_Media'] = df_min['Global_reactive_power'].resample('D').mean()

print(f"Série diária agregada: {len(df_diario)} dias ({df_diario['Energia_Total_kWh'].isna().sum()} dias sem leitura completa).")

In [ ]:
def construir_features_causais(df_diario):
    dados = pd.DataFrame(index=df_diario.index)
    
    # Variável Alvo (y): Consumo total do dia t (em kWh/dia)
    dados['Target_Energia_kWh'] = df_diario['Energia_Total_kWh']
    
    # --- GRUPO A: Lags Autoregressivos do Consumo Total (estritamente <= t-1) ---
    dados['lag_1'] = df_diario['Energia_Total_kWh'].shift(1)
    dados['lag_2'] = df_diario['Energia_Total_kWh'].shift(2)
    dados['lag_3'] = df_diario['Energia_Total_kWh'].shift(3)
    dados['lag_7'] = df_diario['Energia_Total_kWh'].shift(7)
    
    # --- GRUPO B: Lags por Circuito e Variáveis Elétricas de Ontem (t-1) ---
    dados['sub1_cozinha_lag1'] = df_diario['Sub1_Cozinha_kWh'].shift(1)
    dados['sub2_lavandaria_lag1'] = df_diario['Sub2_Lavandaria_kWh'].shift(1)
    dados['sub3_climatizacao_lag1'] = df_diario['Sub3_Climatizacao_kWh'].shift(1)
    dados['sub_outros_lag1'] = df_diario['Sub_Outros_kWh'].shift(1)
    dados['energia_variavel_lag1'] = df_diario['Energia_Variavel_kWh'].shift(1)
    dados['voltage_lag1'] = df_diario['Voltage_Media'].shift(1)
    dados['reactive_lag1'] = df_diario['Reactive_Media'].shift(1)
    
    # --- GRUPO C: Estatísticas Móveis sobre o Passado (shift(1) antes do rolling!) ---
    serie_passada = df_diario['Energia_Total_kWh'].shift(1)
    dados['roll_mean_3'] = serie_passada.rolling(window=3, min_periods=2).mean()
    dados['roll_mean_7'] = serie_passada.rolling(window=7, min_periods=5).mean()
    dados['roll_median_7'] = serie_passada.rolling(window=7, min_periods=5).median()
    dados['roll_std_7'] = serie_passada.rolling(window=7, min_periods=5).std()
    dados['roll_mean_14'] = serie_passada.rolling(window=14, min_periods=10).mean()
    
    # --- GRUPO D: Variáveis Determinísticas de Calendário para o Dia a Prever (t) ---
    # Nota: O dataset UCI (Sceaux, Paris) corresponde a uma habitação em França -> holidays.France()
    anos_dataset = list(range(df_diario.index.year.min(), df_diario.index.year.max() + 1))
    feriados_fr = holidays.France(years=anos_dataset)
    
    dados['dia_semana'] = dados.index.dayofweek
    dados['is_fim_de_semana'] = (dados.index.dayofweek >= 5).astype(int)
    dados['mes'] = dados.index.month
    dados['is_feriado'] = dados.index.normalize().isin(feriados_fr).astype(int)
    dados['mes_sin'] = np.sin(2 * np.pi * dados['mes'] / 12.0)
    dados['mes_cos'] = np.cos(2 * np.pi * dados['mes'] / 12.0)
    
    # Remover linhas iniciais de aquecimento das janelas e dias afetados pelos NaNs originais
    dados_validos = dados.dropna().copy()
    
    # Verificação de integridade temporal (asserts)
    assert dados_validos.index.is_monotonic_increasing, "Índice temporal não está ordenado cronologicamente."
    assert not dados_validos.isna().any().any(), "Existem valores NaN residuais na matriz de modelação."
    
    X = dados_validos.drop(columns=['Target_Energia_kWh'])
    y = dados_validos['Target_Energia_kWh']
    assert X.shape[1] >= 12, f"Número insuficiente de variáveis ({X.shape[1]} < 12)."
    
    return X, y, dados_validos

X, y, df_model = construir_features_causais(df_diario)
print(f"Matriz causal construída com sucesso: {X.shape[0]} dias válidos x {X.shape[1]} variáveis candidatas.")
print("\nLista das 22 variáveis candidatas:")
for i, col in enumerate(X.columns, 1):
    print(f" {i:2d}. {col}")
display(X.head(3))

## 3.2 Seleção Comparada dentro de `Pipeline` e Validação Temporal (`TimeSeriesSplit`)

### Porque é que o `KFold` aleatório constituiria *Data Leakage* aqui?
1. **Autocorrelação e sobreposição de janelas:** Como as variáveis incluem *lags* (`lag_1`, `lag_2`, `lag_7`) e médias móveis (`roll_mean_7`, `roll_mean_14`), dias adjacentes partilham quase toda a informação histórica. Numa divisão aleatória (`KFold(shuffle=True)`), o modelo usaria observações dos dias $t+1$ e $t+2$ no conjunto de treino para prever o dia $t$ no conjunto de teste — permitindo-lhe interpolar o futuro em vez de o prever.
2. **Fuga no pré-processamento e na seleção de variáveis:** O `StandardScaler` e os algoritmos de seleção (Informação Mútua, RFE, Lasso) aprenderiam médias, variâncias e pesos a partir de períodos futuros (por exemplo, ondas de frio do inverno seguinte), inflacionando artificialmente as métricas de erro.
3. **Solução (`TimeSeriesSplit` + `Pipeline`):** Utilizamos `TimeSeriesSplit(n_splits=5)` em janela expansiva (*expanding window*), garantindo que em cada *fold* o treino ocorre estritamente no passado ($1 \dots T_k$) e a validação no futuro imediato ($T_k+1 \dots T_{k+1}$), com todo o escalonamento e seleção encapsulados dentro do `Pipeline`.

In [ ]:
# Configuração da validação cruzada estritamente temporal (5 folds em janela expansiva)
tscv = TimeSeriesSplit(n_splits=5)
K_FEATURES = 6  # Número alvo de variáveis a reter nos métodos Filter e Wrapper

# Definir os 4 Pipelines (1 Baseline com todas as variáveis + 3 métodos de seleção)
mi_scorer = partial(mutual_info_regression, random_state=42)

pipelines = {
    'Todas as Variáveis (Baseline RL)': Pipeline([
        ('scaler', StandardScaler()),
        ('regressor', LinearRegression())
    ]),
    'Filter (SelectKBest - Info Mútua)': Pipeline([
        ('scaler', StandardScaler()),
        ('selector', SelectKBest(score_func=mi_scorer, k=K_FEATURES)),
        ('regressor', LinearRegression())
    ]),
    'Wrapper (RFE - Regressão Linear)': Pipeline([
        ('scaler', StandardScaler()),
        ('selector', RFE(estimator=LinearRegression(), n_features_to_select=K_FEATURES, step=1)),
        ('regressor', LinearRegression())
    ]),
    'Embedded (SelectFromModel - LassoCV)': Pipeline([
        ('scaler', StandardScaler()),
        ('selector', SelectFromModel(
            LassoCV(alphas=np.logspace(-2, 0.3, 30), cv=TimeSeriesSplit(n_splits=3), random_state=42, max_iter=10000)
        )),
        ('regressor', LinearRegression())
    ])
}

# Executar avaliação temporal e registar estabilidade das variáveis entre folds
resultados_folds = []
historico_selecao = {
    nome: pd.DataFrame(0, index=X.columns, columns=[f'Fold_{i+1}' for i in range(5)])
    for nome in pipelines if 'Todas' not in nome
}

# Também avaliamos a Baseline Ingénua de Persistência (Previsão = consumo de ontem, lag_1)
for fold_idx, (train_idx, val_idx) in enumerate(tscv.split(X)):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    # Referência ingénua: Persistência (y_hat[t] = lag_1[t])
    y_pred_naive = X_val['lag_1']
    resultados_folds.append({
        'Metodo': 'Persistência Ingénua (Ontem)',
        'Fold': fold_idx + 1,
        'N_Vars': 1,
        'MAE_kWh': mean_absolute_error(y_val, y_pred_naive),
        'RMSE_kWh': root_mean_squared_error(y_val, y_pred_naive),
        'R2': r2_score(y_val, y_pred_naive)
    })
    
    for nome, pipe in pipelines.items():
        pipe.fit(X_train, y_train)
        y_pred = pipe.predict(X_val)
        
        if 'selector' in pipe.named_steps:
            suporte = pipe.named_steps['selector'].get_support()
            historico_selecao[nome].iloc[:, fold_idx] = suporte.astype(int)
            n_vars = int(suporte.sum())
        else:
            n_vars = X.shape[1]
            
        resultados_folds.append({
            'Metodo': nome,
            'Fold': fold_idx + 1,
            'N_Vars': n_vars,
            'MAE_kWh': mean_absolute_error(y_val, y_pred),
            'RMSE_kWh': root_mean_squared_error(y_val, y_pred),
            'R2': r2_score(y_val, y_pred)
        })

df_res = pd.DataFrame(resultados_folds)

# Resumo agregado por método (Média e Desvio Padrão entre os 5 folds temporais)
tabela_comparativa = df_res.groupby('Metodo', sort=False).agg(
    Vars_Medias=('N_Vars', 'mean'),
    MAE_Media=('MAE_kWh', 'mean'),
    MAE_Std=('MAE_kWh', 'std'),
    RMSE_Media=('RMSE_kWh', 'mean'),
    RMSE_Std=('RMSE_kWh', 'std'),
    R2_Medio=('R2', 'mean')
).round(3)

print("=== Desempenho em Validação Temporal (TimeSeriesSplit - 5 Folds) ===")
display(tabela_comparativa)

In [ ]:
# Análise de Estabilidade entre Folds (Frequência de Seleção e Índice de Jaccard)
def calcular_jaccard_medio(df_bin):
    n_folds = df_bin.shape[1]
    jaccards = []
    for i in range(n_folds):
        for j in range(i + 1, n_folds):
            s1 = set(df_bin.index[df_bin.iloc[:, i] == 1])
            s2 = set(df_bin.index[df_bin.iloc[:, j] == 1])
            uniao = len(s1 | s2)
            jaccards.append(len(s1 & s2) / uniao if uniao > 0 else 1.0)
    return float(np.mean(jaccards))

resumo_estabilidade = []
df_freq_geral = pd.DataFrame(index=X.columns)

for nome, df_bin in historico_selecao.items():
    freq_pct = (df_bin.sum(axis=1) / df_bin.shape[1]) * 100
    df_freq_geral[nome] = freq_pct
    jaccard = calcular_jaccard_medio(df_bin)
    vars_estaveis = list(freq_pct[freq_pct >= 60].index)
    resumo_estabilidade.append({
        'Metodo': nome,
        'Indice_Jaccard_Medio': round(jaccard, 3),
        'Vars_Estaveis_(>=60%_folds)': ', '.join(vars_estaveis) if vars_estaveis else 'Nenhuma'
    })

print("=== Estabilidade da Seleção entre Folds (Índice de Jaccard: 1.0 = seleção idêntica em todos os folds) ===")
display(pd.DataFrame(resumo_estabilidade))

print("=== Frequência de Seleção por Variável (% de Folds em que foi escolhida) ===")
display(df_freq_geral.loc[df_freq_geral.sum(axis=1) > 0].sort_values(by=list(df_freq_geral.columns), ascending=False))

In [ ]:
# FIGURA 8: Estabilidade da Seleção (Heatmap) e Comparação de Erro MAE (Todas vs. Selecionadas)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15.5, 6.2), gridspec_kw={'width_ratios': [1.25, 1.0]})

# Painel Esquerdo: Heatmap de Frequência de Seleção (%)
df_plot_freq = df_freq_geral.loc[df_freq_geral.sum(axis=1) > 0].copy()
df_plot_freq.columns = ['Filter\n(Info Mútua)', 'Wrapper\n(RFE)', 'Embedded\n(LassoCV)']
df_plot_freq = df_plot_freq.sort_values(by=list(df_plot_freq.columns), ascending=False)

sns.heatmap(
    df_plot_freq, ax=ax1, cmap='YlGnBu', annot=True, fmt='.0f',
    vmin=0, vmax=100, linewidths=0.6, linecolor='white',
    cbar_kws={'label': 'Frequência de Seleção nos 5 Folds (%)'}
)
ax1.set_title('Estabilidade: Lags de 1 e 7 dias, médias móveis e climatização dominam a seleção',
              fontsize=11.5, fontweight='bold', loc='left', pad=12)
ax1.set_ylabel('Variáveis Candidatas')
ax1.set_xlabel('Método de Seleção (dentro do Pipeline)')

# Painel Direito: Comparação de MAE Médio entre Folds (Eixo Y iniciado a 0!)
metodos_plot = [
    'Persistência Ingénua (Ontem)',
    'Todas as Variáveis (Baseline RL)',
    'Filter (SelectKBest - Info Mútua)',
    'Wrapper (RFE - Regressão Linear)',
    'Embedded (SelectFromModel - LassoCV)'
]
labels_curtos = ['Persistência\n(1 var)', 'RL Todas\n(22 vars)', 'RL Filter\n(6 vars)', 'RL Wrapper\n(6 vars)', 'RL Embedded\n(LassoCV)']
maes = [tabela_comparativa.loc[m, 'MAE_Media'] for m in metodos_plot]
stds = [tabela_comparativa.loc[m, 'MAE_Std'] for m in metodos_plot]
cores_barras = ['#95a5a6', '#34495e', '#2980b9', '#16a085', '#d35400']

bars = ax2.bar(labels_curtos, maes, yerr=stds, capsize=4, color=cores_barras, width=0.58, edgecolor='black', linewidth=0.6)
ax2.set_ylim(0, max(maes) * 1.28)
for bar, val in zip(bars, maes):
    ax2.text(bar.get_x() + bar.get_width()/2, val + 0.15, f'{val:.2f}', ha='center', va='bottom', fontsize=9.5, fontweight='bold')

ax2.set_title('Modelos parcimoniosos (4–6 variáveis) igualam ou superam a regressão com todas as 22 variáveis',
              fontsize=11.5, fontweight='bold', loc='left', pad=12)
ax2.set_ylabel('Erro Absoluto Médio — MAE (kWh/dia)')
ax2.set_xlabel('Abordagem de Modelação (TimeSeriesSplit)')
ax2.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# FIGURA 9: Série Real vs. Prevista no Último Fold Temporal (Teste Out-of-Sample Futuro)
ultimo_train_idx, ultimo_val_idx = list(tscv.split(X))[-1]
X_tr_last, y_tr_last = X.iloc[ultimo_train_idx], y.iloc[ultimo_train_idx]
X_va_last, y_va_last = X.iloc[ultimo_val_idx], y.iloc[ultimo_val_idx]

pipe_embedded = pipelines['Embedded (SelectFromModel - LassoCV)']
pipe_embedded.fit(X_tr_last, y_tr_last)
y_pred_last = pd.Series(pipe_embedded.predict(X_va_last), index=y_va_last.index)

vars_finais = list(X.columns[pipe_embedded.named_steps['selector'].get_support()])
print(f"Variáveis finais retidas pelo Embedded (LassoCV) no último bloco temporal ({len(vars_finais)} vars):")
print(vars_finais)

fig, ax = plt.subplots(figsize=(13, 4.6))
ax.plot(y_va_last.index, y_va_last.values, color='#2c3e50', linewidth=1.8, marker='o', markersize=3.5, label='Consumo Real Observado (kWh/dia)')
ax.plot(y_pred_last.index, y_pred_last.values, color='#d35400', linewidth=2.0, linestyle='--', label='Previsão RL + Seleção Embedded (kWh/dia)')
ax.axhline(ENERGIA_BASAL_KWH_DIA, color='#c0392b', linestyle=':', linewidth=1.4, label=f'Carga Basal Constante ({ENERGIA_BASAL_KWH_DIA:.1f} kWh/dia)')

mae_last = mean_absolute_error(y_va_last, y_pred_last)
ax.set_title(f'Previsão out-of-sample (último fold temporal): modelo acompanha a tendência semanal e térmica (MAE = {mae_last:.2f} kWh/dia)',
             fontsize=12, fontweight='bold', loc='left', pad=12)
ax.set_xlabel('Data')
ax.set_ylabel('Energia Diária (kWh/dia)')
ax.set_ylim(bottom=0)
ax.legend(loc='upper left', frameon=True)
plt.tight_layout()
plt.show()

### Conclusão e Recomendação Estratégica para a LusoEnergia

1. **Comparação entre Métodos de Seleção e Estabilidade:**
   - **Filter (`SelectKBest` — Informação Mútua):** Apresenta elevada estabilidade (alto índice de Jaccard) porque avalia relações marginais não-lineares com o alvo, selecionando consistentemente `roll_mean_7`, `roll_median_7`, `roll_mean_14`, `lag_1`, `lag_7` e `sub3_climatizacao_lag1`. Contudo, como não penaliza redundância entre variáveis correlacionadas, retém várias médias móveis em simultâneo.
   - **Wrapper (`RFE`):** Procura combinações conjuntas que minimizam o erro da regressão linear, mas sofre maior instabilidade entre *folds* iniciais (menor índice de Jaccard) devido à colinearidade entre `lag_1`, `energia_variavel_lag1` e `roll_mean_3`.
   - **Embedded (`LassoCV` dentro de `Pipeline`):** Oferece o melhor equilíbrio entre parcimónia, estabilidade e generalização temporal — anula automaticamente variáveis redundantes colineares e mantém apenas os sinais estruturais essenciais (~4 a 6 variáveis), alcançando um MAE igual ou inferior ao modelo completo de 22 variáveis com menos de um quarto da complexidade.

2. **Que variáveis recomendamos à LusoEnergia medir e manter (e porquê)?**
   - **Manter obrigatoriamente (Custo Zero de Hardware):**
     1. **Leitura diária agregada do contador inteligente (`lag_1`, `lag_7` e `roll_mean_7` / `roll_median_7`):** Capturam a inércia diária imediata, o ciclo semanal (dia útil vs. fim de semana) e o regime térmico recente da habitação.
     2. **Variáveis de calendário (`is_fim_de_semana` / `dia_semana` e `mes_cos` / `mes`):** Permitem antecipar saltos de ocupação ao fim de semana e o ciclo anual sem qualquer sensor adicional.
   - **Medir seletivamente (Alto Retorno Operacional):**
     3. **Sub-medição dedicada à climatização e águas quentes (`Sub_metering_3`):** Entre os três sub-circuitos físicos, é o único com impacto consistente e persistente na previsão do dia seguinte, pois reflete a carga térmica acumulada (termoacumulador e aquecimento).
   - **Dispensar para previsão agregada de 24h:**
     4. **Sub-contadores individuais de cozinha (`Sub_1`) e lavandaria (`Sub_2`):** Como refletem eventos pontuais e esporádicos (ligar o forno ou a máquina de lavar num dia específico), introduzem ruído na regressão diária e são sistematicamente descartados pelo Lasso/RFE (devendo ser mantidos apenas se a LusoEnergia quiser oferecer serviços de deteção de férias ou alertas de eletrodomésticos).